# Evaluare Preliminară Modele Locale
## RoMistral-7b-Instruct vs. RoGemma-7b-Instruct

Acest notebook evaluează acuratețea celor două modele locale pe un subset de 20 de conversații reprezentative din dataset-ul de 100 de conversații.

**Sarcini evaluate:**
- Detectarea intenției clientului
- Estimarea satisfacției clientului

**Metrici calculate:**
- Accuracy
- F1 Score (macro)
- Timp de inferență per apel

## 1. Instalare dependințe

In [1]:
!pip install transformers torch accelerate sentencepiece scikit-learn pandas matplotlib seaborn -q


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## 2. Importuri și configurare

In [2]:
import json
import os
import time
import random
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import warnings
warnings.filterwarnings('ignore')

print('Importuri OK')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {"MPS (Apple Silicon)" if torch.backends.mps.is_available() else "CPU"}')

/Users/antoniadumitru/Desktop/facultate/Disertatie/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Importuri OK
PyTorch version: 2.10.0
Device: MPS (Apple Silicon)


## 3. Selectare subset reprezentativ de conversații

In [ ]:
# Calea catre folderul cu conversatii adnotate
# Modifica daca folderul tau e in alta locatie
FOLDER_ADNOTAT = './conversatii_adnotate'

def incarca_toate_conversatiile(folder):
    """Incarca toate conversatiile adnotate din toate domeniile."""
    conversatii = []
    for domeniu in os.listdir(folder):
        domeniu_path = os.path.join(folder, domeniu)
        if not os.path.isdir(domeniu_path):
            continue
        for fisier in os.listdir(domeniu_path):
            if fisier.endswith('.json'):
                with open(os.path.join(domeniu_path, fisier), encoding='utf-8') as f:
                    conversatii.append(json.load(f))
    return conversatii

def selecteaza_subset_reprezentativ(conversatii, n=20):
    """Selecteaza un subset echilibrat: proportional din fiecare domeniu si satisfactie."""
    random.seed(42)  # Reproducibilitate
    
    # Grupeaza pe satisfactie
    grupuri = {'pozitiv': [], 'neutru': [], 'negativ': []}
    for conv in conversatii:
        sat = conv.get('satisfactie', 'neutru')
        if sat in grupuri:
            grupuri[sat].append(conv)
    
    # Selecteaza proportional: ~3 pozitive, ~5 neutre, ~12 negative
    subset = []
    subset += random.sample(grupuri['pozitiv'], min(3, len(grupuri['pozitiv'])))
    subset += random.sample(grupuri['neutru'], min(5, len(grupuri['neutru'])))
    subset += random.sample(grupuri['negativ'], min(12, len(grupuri['negativ'])))
    
    random.shuffle(subset)
    return subset[:n]

# Incarca si selecteaza
toate = incarca_toate_conversatiile(FOLDER_ADNOTAT)
subset = selecteaza_subset_reprezentativ(toate, n=20)

print(f'Total conversatii disponibile: {len(toate)}')
print(f'Subset selectat: {len(subset)} conversatii')

# Afiseaza distributia
distributie = {}
for conv in subset:
    sat = conv.get('satisfactie', 'necunoscut')
    distributie[sat] = distributie.get(sat, 0) + 1
print(f'Distributie satisfactie in subset: {distributie}')

distributie_domeniu = {}
for conv in subset:
    dom = conv.get('domeniu', 'necunoscut')
    distributie_domeniu[dom] = distributie_domeniu.get(dom, 0) + 1
print(f'Distributie domenii in subset: {distributie_domeniu}')

## 4. Definire prompturi rafinate

In [ ]:
INTENTII_PER_DOMENIU = {
    'banking': [
        'verificare_sold', 'blocare_card', 'transfer_fonduri', 'informatii_credit',
        'reclamatie_tranzactie', 'programare_consilier', 'activare_card',
        'schimb_valutar', 'resetare_pin', 'acces_homebanking', 'alta_solicitare'
    ],
    'medicina': [
        'programare_consultatie', 'anulare_programare', 'reprogramare',
        'rezultate_analize', 'informatii_medic', 'reteta_medicala',
        'urgenta_medicala', 'sesizare_personal', 'alta_solicitare'
    ],
    'retail': [
        'status_comanda', 'retur_produs', 'reclamatie_produs', 'informatii_stoc',
        'modificare_comanda', 'anulare_comanda', 'problema_livrare',
        'informatii_garantie', 'alta_solicitare'
    ],
    'telecom': [
        'activare_abonament', 'problema_semnal', 'factura_neasteptata',
        'schimbare_abonament', 'portare_numar', 'reziliere_contract',
        'problema_internet', 'roaming', 'alta_solicitare'
    ],
    'servicii_publice': [
        'informatii_acte', 'programare_ghiseu', 'sesizare_problema',
        'status_dosar', 'contestatie', 'plata_taxe',
        'informatii_program', 'reclamatie_serviciu', 'alta_solicitare'
    ]
}

def get_prompt_intentie(dialog, domeniu):
    intentii = INTENTII_PER_DOMENIU.get(domeniu, INTENTII_PER_DOMENIU['banking'])
    return f"""Ești un expert în analiza conversațiilor telefonice.
Identifică EXCLUSIV intențiile clientului din conversația de mai jos.

REGULI:
- Include DOAR ce a vrut sau cerut clientul, nu acțiunile operatorului
- Dacă operatorul a făcut ceva din proprie inițiativă, NU include acea intenție
- Folosește alta_solicitare DOAR dacă nicio intenție din listă nu se potrivește

INTENȚII DISPONIBILE: {', '.join(intentii)}

CONVERSAȚIE:
{dialog}

Răspunde DOAR cu intenția principală, un singur cuvânt din lista de mai sus:"""


def get_prompt_satisfactie(dialog):
    return f"""Ești un expert în analiza satisfacției clienților.
Analizează conversația și determină nivelul de satisfacție al clientului la final.

DEFINIȚII:
- pozitiv: problema rezolvată complet, clientul mulțumit și o exprimă
- neutru: problema rezolvată tehnic dar clientul pleacă indiferent sau ușor nemulțumit de proces
- negativ: clientul pleacă cu problema nerezolvată SAU cu frustrare clară față de rezultat

REGULI:
- Un singur comentariu negativ urmat de acceptare NU înseamnă automat negativ
- Uită-te la TONUL GENERAL și REZULTATUL FINAL al conversației
- Frustrarea poate fi implicită: ironie, resemnare, remarci tăioase

CONVERSAȚIE:
{dialog}

Răspunde DOAR cu unul dintre cuvintele: pozitiv, neutru, negativ"""


print('Prompturi definite OK')

## 5. Funcții de inferență

In [ ]:
def incarca_model(model_name):
    """Incarca tokenizer si model."""
    print(f'Se incarca {model_name}...')
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Foloseste MPS daca e disponibil (Apple Silicon), altfel CPU
    device = 'mps' if torch.backends.mps.is_available() else 'cpu'
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16 if device == 'mps' else torch.float32
    ).to(device)
    
    print(f'Model incarcat pe {device}')
    return tokenizer, model, device


def genereaza_raspuns(tokenizer, model, device, prompt, max_new_tokens=30):
    """Genereaza un raspuns si masoara latenta."""
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1024).to(device)
    
    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id
        )
    latenta = time.time() - start
    
    # Decodifica doar tokenii noi (nu promptul)
    input_length = inputs['input_ids'].shape[1]
    new_tokens = outputs[0][input_length:]
    raspuns = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    
    return raspuns, latenta


def extrage_eticheta_intentie(raspuns, domeniu):
    """Extrage eticheta de intentie din raspunsul modelului."""
    intentii = INTENTII_PER_DOMENIU.get(domeniu, [])
    raspuns_lower = raspuns.lower().strip()
    
    # Cauta prima intentie mentionata in raspuns
    for intentie in intentii:
        if intentie in raspuns_lower:
            return intentie
    
    return 'alta_solicitare'


def extrage_eticheta_satisfactie(raspuns):
    """Extrage eticheta de satisfactie din raspunsul modelului."""
    raspuns_lower = raspuns.lower().strip()
    
    if 'pozitiv' in raspuns_lower:
        return 'pozitiv'
    elif 'negativ' in raspuns_lower:
        return 'negativ'
    elif 'neutru' in raspuns_lower:
        return 'neutru'
    
    return 'neutru'  # Default


print('Functii definite OK')

## 6. Evaluare RoMistral

In [ ]:
# Incarca RoMistral
tokenizer_mistral, model_mistral, device = incarca_model('OpenLLM-Ro/RoMistral-7b-Instruct')

In [ ]:
rezultate_mistral = []

for i, conv in enumerate(subset):
    conv_id = conv['id']
    domeniu = conv.get('domeniu', 'banking')
    dialog = '\n'.join([f"{r['rol'].upper()}: {r['text']}" for r in conv['conversatie']])
    
    print(f'[{i+1:02d}/20] {conv_id}', end='', flush=True)
    
    # --- Intentie ---
    prompt_int = get_prompt_intentie(dialog, domeniu)
    raspuns_int, latenta_int = genereaza_raspuns(tokenizer_mistral, model_mistral, device, prompt_int)
    intentie_pred = extrage_eticheta_intentie(raspuns_int, domeniu)
    
    # --- Satisfactie ---
    prompt_sat = get_prompt_satisfactie(dialog)
    raspuns_sat, latenta_sat = genereaza_raspuns(tokenizer_mistral, model_mistral, device, prompt_sat)
    satisfactie_pred = extrage_eticheta_satisfactie(raspuns_sat)
    
    # Ground truth
    intentie_gold = conv.get('intentie_gold', ['alta_solicitare'])
    if isinstance(intentie_gold, list):
        intentie_gold = intentie_gold[0]  # Prima intentie ca referinta
    satisfactie_gold = conv.get('satisfactie', 'neutru')
    
    rezultate_mistral.append({
        'id': conv_id,
        'domeniu': domeniu,
        'intentie_gold': intentie_gold,
        'intentie_pred': intentie_pred,
        'intentie_corecta': intentie_pred == intentie_gold,
        'satisfactie_gold': satisfactie_gold,
        'satisfactie_pred': satisfactie_pred,
        'satisfactie_corecta': satisfactie_pred == satisfactie_gold,
        'latenta_intentie': round(latenta_int, 2),
        'latenta_satisfactie': round(latenta_sat, 2),
        'latenta_totala': round(latenta_int + latenta_sat, 2)
    })
    
    print(f' | intentie: {intentie_pred} ({"✓" if intentie_pred == intentie_gold else "✗"}) | satisfactie: {satisfactie_pred} ({"✓" if satisfactie_pred == satisfactie_gold else "✗"}) | {latenta_int+latenta_sat:.0f}s')

df_mistral = pd.DataFrame(rezultate_mistral)
print('\nEvaluare RoMistral finalizata!')

## 7. Evaluare RoGemma

In [ ]:
# Elibereaza memoria inainte sa incarci RoGemma
del model_mistral
del tokenizer_mistral
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

# Incarca RoGemma
tokenizer_gemma, model_gemma, device = incarca_model('OpenLLM-Ro/RoGemma-7b-Instruct')

In [ ]:
rezultate_gemma = []

for i, conv in enumerate(subset):
    conv_id = conv['id']
    domeniu = conv.get('domeniu', 'banking')
    dialog = '\n'.join([f"{r['rol'].upper()}: {r['text']}" for r in conv['conversatie']])
    
    print(f'[{i+1:02d}/20] {conv_id}', end='', flush=True)
    
    # --- Intentie ---
    prompt_int = get_prompt_intentie(dialog, domeniu)
    raspuns_int, latenta_int = genereaza_raspuns(tokenizer_gemma, model_gemma, device, prompt_int)
    intentie_pred = extrage_eticheta_intentie(raspuns_int, domeniu)
    
    # --- Satisfactie ---
    prompt_sat = get_prompt_satisfactie(dialog)
    raspuns_sat, latenta_sat = genereaza_raspuns(tokenizer_gemma, model_gemma, device, prompt_sat)
    satisfactie_pred = extrage_eticheta_satisfactie(raspuns_sat)
    
    # Ground truth
    intentie_gold = conv.get('intentie_gold', ['alta_solicitare'])
    if isinstance(intentie_gold, list):
        intentie_gold = intentie_gold[0]
    satisfactie_gold = conv.get('satisfactie', 'neutru')
    
    rezultate_gemma.append({
        'id': conv_id,
        'domeniu': domeniu,
        'intentie_gold': intentie_gold,
        'intentie_pred': intentie_pred,
        'intentie_corecta': intentie_pred == intentie_gold,
        'satisfactie_gold': satisfactie_gold,
        'satisfactie_pred': satisfactie_pred,
        'satisfactie_corecta': satisfactie_pred == satisfactie_gold,
        'latenta_intentie': round(latenta_int, 2),
        'latenta_satisfactie': round(latenta_sat, 2),
        'latenta_totala': round(latenta_int + latenta_sat, 2)
    })
    
    print(f' | intentie: {intentie_pred} ({"✓" if intentie_pred == intentie_gold else "✗"}) | satisfactie: {satisfactie_pred} ({"✓" if satisfactie_pred == satisfactie_gold else "✗"}) | {latenta_int+latenta_sat:.0f}s')

df_gemma = pd.DataFrame(rezultate_gemma)
print('\nEvaluare RoGemma finalizata!')

## 8. Calcul metrici

In [ ]:
def calculeaza_metrici(df, nume_model):
    """Calculeaza accuracy si F1 pentru intentie si satisfactie."""
    print(f'\n=== {nume_model} ===')
    
    # Intentie
    acc_int = accuracy_score(df['intentie_gold'], df['intentie_pred'])
    f1_int = f1_score(df['intentie_gold'], df['intentie_pred'], average='macro', zero_division=0)
    
    # Satisfactie
    acc_sat = accuracy_score(df['satisfactie_gold'], df['satisfactie_pred'])
    f1_sat = f1_score(df['satisfactie_gold'], df['satisfactie_pred'], average='macro', zero_division=0)
    
    # Latenta
    latenta_medie = df['latenta_totala'].mean()
    
    print(f'  Intentie   — Accuracy: {acc_int:.2%} | F1: {f1_int:.2%}')
    print(f'  Satisfactie — Accuracy: {acc_sat:.2%} | F1: {f1_sat:.2%}')
    print(f'  Latenta medie per conversatie: {latenta_medie:.1f}s')
    
    return {
        'model': nume_model,
        'intentie_accuracy': round(acc_int, 4),
        'intentie_f1': round(f1_int, 4),
        'satisfactie_accuracy': round(acc_sat, 4),
        'satisfactie_f1': round(f1_sat, 4),
        'latenta_medie': round(latenta_medie, 1)
    }


metrici_mistral = calculeaza_metrici(df_mistral, 'RoMistral-7b')
metrici_gemma = calculeaza_metrici(df_gemma, 'RoGemma-7b')

# Tabel comparativ
df_metrici = pd.DataFrame([metrici_mistral, metrici_gemma])
df_metrici = df_metrici.set_index('model')
print('\n=== TABEL COMPARATIV ===')
print(df_metrici.to_string())

## 9. Vizualizare rezultate

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Evaluare Preliminară: RoMistral vs. RoGemma', fontsize=14, fontweight='bold')

modele = ['RoMistral-7b', 'RoGemma-7b']
culori = ['#2196F3', '#4CAF50']

# Grafic 1: Accuracy intentie
valori_acc_int = [metrici_mistral['intentie_accuracy'], metrici_gemma['intentie_accuracy']]
axes[0].bar(modele, valori_acc_int, color=culori)
axes[0].set_title('Accuracy — Detecție Intenție')
axes[0].set_ylim(0, 1)
axes[0].set_ylabel('Accuracy')
for j, v in enumerate(valori_acc_int):
    axes[0].text(j, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Grafic 2: Accuracy satisfactie
valori_acc_sat = [metrici_mistral['satisfactie_accuracy'], metrici_gemma['satisfactie_accuracy']]
axes[1].bar(modele, valori_acc_sat, color=culori)
axes[1].set_title('Accuracy — Estimare Satisfacție')
axes[1].set_ylim(0, 1)
axes[1].set_ylabel('Accuracy')
for j, v in enumerate(valori_acc_sat):
    axes[1].text(j, v + 0.02, f'{v:.1%}', ha='center', fontweight='bold')

# Grafic 3: Latenta medie
valori_latenta = [metrici_mistral['latenta_medie'], metrici_gemma['latenta_medie']]
axes[2].bar(modele, valori_latenta, color=culori)
axes[2].set_title('Latență Medie per Conversație')
axes[2].set_ylabel('Secunde')
for j, v in enumerate(valori_latenta):
    axes[2].text(j, v + 1, f'{v:.1f}s', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('evaluare_modele_locale.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grafic salvat: evaluare_modele_locale.png')

## 10. Analiza erorilor

In [ ]:
def analiza_erori(df, nume_model):
    """Afiseaza conversatiile clasificate gresit."""
    print(f'\n=== ERORI {nume_model} ===')
    
    erori_int = df[~df['intentie_corecta']]
    erori_sat = df[~df['satisfactie_corecta']]
    
    print(f'\nErori intentie ({len(erori_int)}/{len(df)}):')
    for _, row in erori_int.iterrows():
        print(f'  {row["id"]} | gold: {row["intentie_gold"]} | pred: {row["intentie_pred"]}')
    
    print(f'\nErori satisfactie ({len(erori_sat)}/{len(df)}):')
    for _, row in erori_sat.iterrows():
        print(f'  {row["id"]} | gold: {row["satisfactie_gold"]} | pred: {row["satisfactie_pred"]}')

analiza_erori(df_mistral, 'RoMistral-7b')
analiza_erori(df_gemma, 'RoGemma-7b')

## 11. Salvare rezultate

In [ ]:
# Salveaza rezultatele detaliate
df_mistral.to_csv('rezultate_romistral.csv', index=False)
df_gemma.to_csv('rezultate_rogemma.csv', index=False)

# Salveaza tabelul comparativ
df_metrici.to_csv('metrici_comparative.csv')

print('Fisiere salvate:')
print('  - rezultate_romistral.csv')
print('  - rezultate_rogemma.csv')
print('  - metrici_comparative.csv')
print('  - evaluare_modele_locale.png')